# Appendix C - Annotated Jupyter Notebook

This Jupyter Notebook was used to supported the Computer-informed reconstruction of John Browne's fragmentary Magnificat from the Eton Choirbook (Eton College, MS 178). It has been annotated to explain what each section of code is doing, or what it is able to do. All code cells have been run, and their outputs retained in order to demonstrate a how this notebook might be used.

Once step 0 is run, the other sections can be run independently (although not nesseraily the subsections within these sections).

This notebook allows for the searching of Melodic, Harmonic, and Contrapuntal patterns in an established corpus.

For more information on the code, see https://github.com/HCDigitalScholarship/intervals/tree/main 


## 0.1 Import CRIM Intervals and other code used with CRIM Intervals

In this cell, I load the main libraries and set up the working environment for the rest of the notebook.

 - `crim_intervals` (and its submodules such as `main_objs` and `visualizations`) provides the core tools for working with the CRIM melodic interval data.
 - `pandas`, `re`, and `glob` are used for data handling and basic text and file manipulation.
 - `altair`, `matplotlib`, `seaborn`, and `plotly` are imported for different kinds of visualisation; later plots can use whichever library is most convenient.
 - `pyvis` and `IPython.display` support interactive network diagrams and rich display inside the notebook.
 - `ipywidgets` enables interactive controls (sliders, dropdowns, etc.) for exploring and manipulating the musical data.

The next section sets up a simple directory structure for saving outputs:

 - `saved_csv` is used to store CSV exports generated during the analysis.
 - `Music_Files` is intended for source music files (e.g. MEI, XML, or other encodings).
 - If the folder does not already exist, it is automatically created; otherwise, a message confirms that the folder is already present.

I then configure the plotting backends so that visualisations render correctly inside the notebook:

 - `alt.renderers.enable('default')` activates Altair’s default renderer.
 - The Plotly renderer is set to `"plotly_mimetype+notebook_connected"` to ensure fully interactive figures display properly.

Finally, I define a small helper function, `_convertTuple`, which converts a tuple into a single underscore-separated string (e.g. `('1','-2','2') → '1_-2_2'`). This is used later to convert interval tokens or other symbolic representations into a consistent string format for analysis and comparison.

These libraries are not nesserarily used in the code that follows (such as the data visualisation tools). However, they are installed so that they might be used if required.

In [4]:
import crim_intervals
from crim_intervals import * 
from crim_intervals import main_objs
import crim_intervals.visualizations as viz
import pandas as pd
import re
import altair as alt
import matplotlib.pyplot as plt
import seaborn as sns
# from ipywidgets import interact
# from pandas.io.json import json_normalize
from pyvis.network import Network
from IPython.display import display
import requests
import glob as glob
import os
from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    print("created folder : ", MYDIR)
else:
    print(MYDIR, "folder already exists.")
    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)

else:
    print(MUSDIR, "folder already exists.")

alt.renderers.enable('default')
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"


def _convertTuple(tup):
    out = ""
    if isinstance(tup, tuple):
        out = '_'.join(tup)
    return out

saved_csv folder already exists.
Music_Files folder already exists.


## 0.2 Establish Corpus
This part of the notebook gathers the music files and prepares them so the rest of the project can analyse them.

`glob.glob('Music_Files/Eton_Corpus/*.musicxml')` searches within the `Music_Files/Eton_Corpus` directory and returns a list of all files ending in `.musicxml`. This list is stored in `filenames`. These files are the encoded musical scores used for interval extraction and pattern analysis.

`CorpusBase(filenames)` loads all these music files into a single collection called `corpus`. This object `(corpus)` allows the notebook to treat all pieces together and makes it easy to run the later steps of the analysis.

In [5]:
filenames = glob.glob('Music_Files/Eton_Corpus/*.musicxml')

corpus = CorpusBase(filenames)

## 1.1 Melodic ngrams
This cell takes the music in the corpus and turns it into a large table of melodic patterns. Each row in the final table represents one short melodic pattern in a particular voice, with its composer, title, and position in the piece.

The first line `# corpus = CorpusBase(filtered_file_names[0:10])` is commented out, so it does nothing at the moment. If you remove the `#`, it would build a smaller corpus from the first 10 files in filtered_file_names. You can change 0:10 to any slice (e.g. 0:5, 10:20) to work with a smaller or different sample of pieces. Using only part of the corpus is useful if you wish to test and explore the analysis more quickly before running it on the full dataset. If commented out, this cell assumes that corpus has already been created earlier and contains all the pieces you want to study (see 0.2 Establish Corpus).

Otherwise, the first step is to get the melodic intervals for each piece.

#### Step 1 – Get melodic intervals for each piece

 - `func1 = ImportedPiece.melodic` Here we tell CRIM Intervals which kind of analysis we want to run first: the melodic intervals in each voice.
 - `list_of_dfs = corpus.batch(func=func1, kwargs={'kind': 'd', 'end': False}, metadata=False)`. This line runs the melodic-interval calculation on every piece in the corpus and returns a list of tables (one table per piece).
   - The `kind='d'` setting asks for diatonic intervals, rather than differtiating between, for example, major and minor thirds. You can change this if you want a different type of interval (e.g. chromatic), following the CRIM Intervals options. [add link].
   - `end=False` controls how the calculation treats the very end of the piece. With False, it avoids creating extra “half-intervals” at the end.
   - `metadata=False`. Here we only want the interval data themselves, not the composer/title information yet. That will be added later.
   - `list_of_dfs` This becomes a list of tables, one for each piece, showing the melodic interval at each note position.

#### Step 2 – Turn intervals into short patterns (n-grams)

 - `func2 = ImportedPiece.ngrams` This code chooses the next kind of analysis: building n-grams, i.e. short patterns of consecutive intervals.
 - `list_of_melodic_ngrams = corpus.batch(func=func2, kwargs={'n': 4, 'df': list_of_dfs}, metadata=False)` runs the n-gram calculation across all pieces.
   - `n=4` This tells CRIM Intervals to look for patterns of 4 consecutive intervals. `n` could be changed to any positive integer if you want shorter or longer patterns.
   - `df=list_of_dfs` The function uses the interval data from Step 1 as its starting point
   - `list_of_melodic_ngrams` This is again a list of tables, now with `n`-interval patterns for each piece.

#### Step 3 – Add bar and beat information

 - `func3 = ImportedPiece.detailIndex` This step adds more precise location information for each pattern.
 - `list_of_detail_index = corpus.batch(func=func3, kwargs={'offset': False,'df': list_of_melodic_ngrams}, metadata=False)`
   - `offset=False` This tells CRIM Intervals to give locations in measure and beat numbers (e.g. bar 23, beat 3) rather than a running time count.
   - `df=list_of_melodic_ngrams` This uses the n-gram tables from the previous step as input.
   - `list_of_detail_index` This is another list of tables, now with each `n`-interval pattern tied to where it occurs in the score.

#### Step 4 – Add information about parts/voices

 - `func4 = ImportedPiece.numberParts` Here we make sure each row knows which voice (e.g. cantus, tenor) it belongs to.
 - `list_of_np = corpus.batch(func=func4, kwargs={'df': list_of_detail_index}, metadata=True)`
   - `df=list_of_detail_index` We start from the tables with patterns and locations.
   - `metadata=True` This time, we tell CRIM Intervals to include composer, title, and other header information in the result, so that we can later group and compare pieces.
   - `list_of_np` This is a list of tables, one per piece, now containing: composer, title, date, measure, beat, voice, and the 4-interval patterns.

#### Step 5 – Combine everything into one big table

 - `mel_corpus = pd.concat(list_of_np)` This stacks all the per-piece tables together into a single large table called `mel_corpus`. Each row is one pattern in one voice from one piece.
 - `mel_corpus = mel_corpus.reset_index()` This simply resets the row numbering so it runs from 0, 1, 2, … in a clean way.

#### Step 6 – Reshape so each row is “one pattern in one voice”

Right now, each voice may be in its own column (e.g. Cantus, Altus, Tenor, Bassus). The next lines reshape the table so that each row is: Composer, Title, Date, Measure, Beat, Voice, Ngram

 - `val_vars = [col for col in mel_corpus.columns if col not in ['Composer', 'Title', 'Date', 'Measure', 'Beat']]`
Here we build a list called val_vars.
      - It contains all columns except the “context” columns Composer, Title, Date, Measure, and Beat.
      - In practice, `val_vars` will be the columns that hold patterns for each voice.

You can adjust the list of excluded columns if you add or remove metadata (for example, if you want to keep a field like Genre fixed as well).

 - `mel_corpus_melted = mel_corpus.melt(id_vars = ['Composer', 'Title', 'Date', 'Measure', 'Beat'], value_vars = val_vars).dropna(subset = 'value')`
      - `.melt(...)` turns the table from “many voice columns” into one voice column and one pattern column, so that each row is a single pattern in a single voice.
      - `id_vars` are the columns that stay the same in each row (composer, title, date, bar, beat).
      - `value_vars` (the voices) get turned into two columns: one that says which voice it is, and one that holds the pattern.
      - `.dropna(subset='value')` throws away rows where there is no pattern (empty cells).
 - `mel_corpus_melted = mel_corpus_melted.rename(columns= {'variable': 'Voice', 'value': 'Ngram'})`
This renames the two generic columns created by melt:
      - `variable` → `Voice` (e.g. Cantus, Tenor, etc.)
      - `value` → `Ngram` (the `n`-interval pattern itself)

#### Step 7 – Turn patterns into a simple text format

 - `mel_corpus_melted['Ngram'] = mel_corpus_melted['Ngram'].apply(_convertTuple)`

At this point, each Ngram may be stored as a small tuple, such as `('1', '-2', '-2', '2')`. The helper function `_convertTuple` (defined earlier) turns this into a single string with underscores, e.g.: `('1', '-2', '-2', '2')` → `"1_-2_-2_2"`. This makes patterns easier to store, search for, and compare.

 - `mel_corpus_melted`
The final table has one row per pattern, with:
      - who wrote it (`Composer`)
      - which piece (`Title`, `Date`)
      - where it occurs (`Measure`, `Beat`)
      - in which voice (`Voice`)
      - and the pattern itself (`Ngram`), as a simple string.


In [6]:
# corpus = CorpusBase(filtered_file_names[0:10])
func1 = ImportedPiece.melodic
list_of_dfs = corpus.batch(func=func1, kwargs={'kind': 'q', 'end': False}, metadata=False)
func2 = ImportedPiece.ngrams
list_of_melodic_ngrams = corpus.batch(func=func2, kwargs={'n': 5, 'df': list_of_dfs}, metadata=False)
func3 = ImportedPiece.detailIndex
list_of_detail_index = corpus.batch(func=func3, kwargs={'offset': False,'df': list_of_melodic_ngrams}, metadata=False)
func4 = ImportedPiece.numberParts
list_of_np = corpus.batch(func=func4, kwargs={'df': list_of_detail_index}, metadata=True)
mel_corpus = pd.concat(list_of_np)
mel_corpus = mel_corpus.reset_index()
val_vars = [col for col in mel_corpus.columns if col not in ['Composer', 'Title', 'Date', 'Measure', 'Beat']]
mel_corpus_melted = mel_corpus.melt(id_vars = ['Composer', 'Title', 'Date', 'Measure', 'Beat'], value_vars = val_vars).dropna(subset = 'value')
mel_corpus_melted = mel_corpus_melted.rename(columns= {'variable': 'Voice', 'value': 'Ngram'})
mel_corpus_melted['Ngram'] = mel_corpus_melted['Ngram'].apply(_convertTuple)
mel_corpus_melted

,Composer,Title,Date,Measure,Beat,Voice,Ngram
0,William Horwood,Gaude Flore Virginali,None,1.0,1.0,1,M3_-M2_P4_-M2_-m2
1,William Horwood,Gaude Flore Virginali,None,2.0,1.0,1,-M2_P4_-M2_-m2_-M2
3,William Horwood,Gaude Flore Virginali,None,2.0,2.5,1,P4_-M2_-m2_-M2_-M2
4,William Horwood,Gaude Flore Virginali,None,2.0,3.0,1,-M2_-m2_-M2_-M2_-m2
5,William Horwood,Gaude Flore Virginali,None,3.0,1.0,1,-m2_-M2_-M2_-m2_-M2
...,...,...,...,...,...,...,...
149063,Hugh Kellyk,Gaude Flore Virginali,None,269.0,1.0,7,-M2_-m3_P4_-P5_M2
149066,Hugh Kellyk,Gaude Flore Virginali,None,269.0,3.0,7,-m3_P4_-P5_M2_-M2
149067,Hugh Kellyk,Gaude Flore Virginali,None,269.0,4.0,7,P4_-P5_M2_-M2_-m3
149068,Hugh Kellyk,Gaude Flore Virginali,None,270.0,1.0,7,-P5_M2_-M2_-m3_m3


### 1.1.1 Count how often each pattern appears
- Count the patterns: `value_counts()` counts how many times each melodic pattern (`Ngram`) appears in the entire dataset. This gives a quick picture of which patterns are common and which are rare.
- Turn the counts into a table: The results are converted into a DataFrame so they are easier to read, sort, and work with.
- Calculate percentages: A new column shows what percentage of the total each pattern represents, making it easier to compare their relative frequency.
- Display the results: The final table, `counts_df`, lists every pattern along with how often it occurs and what share of the corpus that represents.

In [7]:
counts = mel_corpus_melted['Ngram'].value_counts()

counts_df = counts.reset_index()
counts_df.columns = ['Ngram', 'Count']

total = counts_df['Count'].sum()
counts_df['Percent'] = (counts_df['Count'] / total * 100).round(2)

counts_df

,Ngram,Count,Percent
0,-M2_-M2_-m2_-M2_-M2,300,0.87
1,M2_-M2_-M2_-m2_-M2,164,0.48
2,M2_M2_-M2_-M2_-m2,148,0.43
3,-m2_-M2_-M2_-M2_-m2,141,0.41
4,-m2_-M2_-M2_-m2_-M2,139,0.41
...,...,...,...
12745,-M2_P1_M2_P5_-M2,1,0.00
12746,P1_M2_P5_-M2_-M2,1,0.00
12747,P4_-M2_-M2_P1_-m3,1,0.00
12748,-P4_P1_M3_-M3_P5,1,0.00


## 1.2 Matching Full Ngram

This cell enables the ability to search for specific melodic patterns. These patterns must of `n` length previously established.

 - The Variable `my_pattern` is defined by the user and stores the exact sequence of intervals you wish to search for in the dataset
 - The line `filtered_df = ... `searches the full table and keeps only the rows where the `Ngram` matches that pattern exactly. This shows every place in the corpus where that pattern ~occurs.
 - `sorted_df_custom = filtered_df.sort_values(
   by=['Title', 'Measure', 'Beat', 'Voice'],
    ascending=[True, True, True, True] `sorts the rows so the occurrences appear in a sensible order
 -  `sorted_df_custom` displays the final lable listing all occurences of the chosen pattern.

In [8]:
my_pattern = 'M2_m2_M2_M2_M2'
filtered_df = mel_corpus_melted[mel_corpus_melted['Ngram'] == my_pattern]

sorted_df_custom = filtered_df.sort_values(
    by=['Title', 'Measure', 'Beat', 'Voice'],
    ascending=[True, True, True, True]
)

sorted_df_custom#.head(100)

,Composer,Title,Date,Measure,Beat,Voice,Ngram
96564,William Horwood\ned./rec. Michael Winter,E37 William Horwood Gaude virgo mater Christi,None,24.0,3.00,5,M2_m2_M2_M2_M2
96860,William Horwood\ned./rec. Michael Winter,E37 William Horwood Gaude virgo mater Christi,None,72.0,3.00,5,M2_m2_M2_M2_M2
113536,William Cornysh,E48 Stabat Mater,None,106.0,1.00,6,M2_m2_M2_M2_M2
134965,William Cornysh,E48 Stabat Mater,None,114.0,1.00,7,M2_m2_M2_M2_M2
114050,William Cornysh,E48 Stabat Mater,None,328.0,3.00,6,M2_m2_M2_M2_M2
24636,Robert Wylkynson,E59 Gaude Virgo Mater Cristi,None,40.0,1.75,2,M2_m2_M2_M2_M2
85152,Not found,E61 Stella Celi,None,41.0,1.00,4,M2_m2_M2_M2_M2
21057,Not found,E61 Stella Celi,None,54.0,2.50,1,M2_m2_M2_M2_M2
85574,Not found,E61 Stella Celi,None,102.0,2.50,4,M2_m2_M2_M2_M2
89978,John Browne,E7 Stabat Virgo Mater Cristi,None,184.0,1.50,5,M2_m2_M2_M2_M2


### 1.2.1 Count occurrences of `my_pattern`

In [9]:
count = filtered_df.shape[0]
print(f"Total occurrences of '{my_pattern}':", count)

Total occurrences of 'M2_m2_M2_M2_M2': 35


## 1.3 Fuzzy Search of Pattern
This block of code looks for musical patterns that are close to a chosen pattern, even if they are not an exact match. This is helpful because musical ideas often appear with small variatoins, and a fuzzy search can reveal related shapes that an exact search would miss

 - Choose a target pattern and tolerance: `my_pattern` is the interval pattern we want to search for, and `threshold` sets how different a pattern is allowed to be while still counting as a match.
 - Turn patterns into lists of numbers: The helper function `parse_pattern` splits a pattern like "2_-2_2_2" into a list of integers. This lets the computer compare interval shapes one step at a time.
 - Measure how similar two patterns are: `levenshtein_intervals` calculates how many small changes (insertions, deletions, or interval adjustments) are needed to turn one pattern into another. Patterns that require fewer changes are treated as more similar.
 - Compute similarity for every pattern in the corpus: The code adds a new column called `distance`, showing how close each pattern in the dataset is to `my_pattern`.
 - Keep only the close matches: Rows with a distance less than or equal to the threshold are kept. These are the patterns that most resemble the target shape.
 - Sort the results: The matches are ordered so that the closest ones appear first, followed by piece, measure, beat, and voice. This makes it easy to see both how similar the pattern is and where it occurs in the music.

The final table, `sorted_df_custom`, shows every place in the corpus where a pattern closely resembling the chosen interval pattern appears.

n.b. this currently only works with searching without quality

In [10]:
import numpy as np
import pandas as pd 

my_pattern = 'M2_m2_M2_M2'
threshold = 2 

# === HELPERS ===
def parse_pattern(s: str):
    """Convert '1_-2_3' -> [1, -2, 3]."""
    return [int(x) for x in s.split('_')]

def levenshtein_intervals(a, b, ins_cost=1, del_cost=1):
    """
    Levenshtein distance on interval sequences.
    Substitution cost = abs(a_i - b_j) so closer intervals are cheaper.
    """
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # base cases
    for i in range(m + 1):
        dp[i][0] = i * del_cost
    for j in range(n + 1):
        dp[0][j] = j * ins_cost

    # dynamic programming
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            sub_cost = abs(a[i - 1] - b[j - 1])
            dp[i][j] = min(
                dp[i - 1][j] + del_cost,        # deletion
                dp[i][j - 1] + ins_cost,        # insertion
                dp[i - 1][j - 1] + sub_cost     # substitution
            )

    return dp[m][n]

# Pre-parse the target pattern once
pattern_tokens = parse_pattern(my_pattern)

def distance_to_pattern(ngram_str: str) -> int:
    """Distance between a corpus Ngram and the target pattern."""
    tokens = parse_pattern(ngram_str)
    return levenshtein_intervals(tokens, pattern_tokens)

# === APPLY TO DATAFRAME ===
# Assumes you already have `mel_corpus_melted` with columns:
# 'Ngram', 'Title', 'Measure', 'Beat', 'Voice'

# Compute distance for each Ngram
mel_corpus_melted['distance'] = mel_corpus_melted['Ngram'].apply(distance_to_pattern)

# Filter to “close” matches
filtered_df = mel_corpus_melted[mel_corpus_melted['distance'] <= threshold]

# Sort: closest patterns first, then your usual ordering
sorted_df_custom = filtered_df.sort_values(
    by=['distance', 'Title', 'Measure', 'Beat', 'Voice'],
    ascending=[True, True, True, True, True]
)

# Look at the result
sorted_df_custom  # or sorted_df_custom.head(50)


ValueError: invalid literal for int() with base 10: 'M2'

## 1.4 Matching Any Substring

This code allows for searching for a shorter interval sequence stored in `my_pattern`. Instead of requiring an exact match, `str.contains()` finds all patterns where the given shape appears as part of a longer pattern. After that, the results are sorted by piece, measure, beat, and voice so it is easy to see where these occurrences happen in the music. `my_pattern`, in this cell, must be shorter than the previously established `n-gram` length.

In [12]:
my_pattern = 'M2_m2_M2'
filtered_df = mel_corpus_melted[mel_corpus_melted['Ngram'].str.contains(my_pattern)]

# Sort by multiple columns
sorted_df_custom = filtered_df.sort_values(
    by=['Title', 'Measure', 'Beat', 'Voice'],
    ascending=[True, True, True, True]  # Sort Measure in descending order
)

sorted_df_custom

,Composer,Title,Date,Measure,Beat,Voice,Ngram
34398,William Cornysh,Ave Maria Mater Dei,None,8.0,3.0,2,P4_-P4_M2_m2_M2
34399,William Cornysh,Ave Maria Mater Dei,None,8.0,4.0,2,-P4_M2_m2_M2_-P4
34400,William Cornysh,Ave Maria Mater Dei,None,9.0,2.0,2,M2_m2_M2_-P4_M2
34415,William Cornysh,Ave Maria Mater Dei,None,11.0,4.5,2,P5_M2_m2_M2_-P4
34417,William Cornysh,Ave Maria Mater Dei,None,12.0,1.0,2,M2_m2_M2_-P4_M2
...,...,...,...,...,...,...,...
98554,John Browne,Stabat mater dolorosa,None,237.0,3.0,5,m3_M2_m2_M2_M2
98555,John Browne,Stabat mater dolorosa,None,237.0,3.5,5,M2_m2_M2_M2_-P4
98567,John Browne,Stabat mater dolorosa,None,239.0,1.0,5,-m3_P1_M2_m2_M2
55772,John Browne,Stabat mater dolorosa,None,239.0,1.5,3,M2_m2_M2_M2_-m3


## 2.1 Corpus Harmonic nGrams

This section uses CRIM Intervals to extract harmonic information from each piece in the corpus, convert it into harmonic n-grams, and reshape the results into a tidy format for analysis.

#### Step 1 — Extract harmonic intervals from each piece
 - Set the harmonic-extraction function.
 - Run it on the entire corpus using `corpus.batch`.
 - Output: a list of DataFrames (one per piece) containing harmonic intervals + metadata.
#### Step 2 — Convert harmonic data into 4-note n-grams
 - Select the n-gram function.
 - Apply it to each harmonic DataFrame, specifying a value for `n`
 - Output: a list of harmonic n-gram DataFrames.
#### Step 3 — Add positional details to each n-gram
 - Use `detailIndex` to attach measure numbers, beat locations, and related timing information.
 - Keeps metadata attached.
#### Step 4 — Number and label musical parts
 - Apply `numberParts` to ensure each n-gram is tied to the correct voice/part in the score.
 - Output: annotated DataFrames for every piece.
#### Step 5 — Combine all pieces into one corpus-wide table
 - Concatenate the list of per-piece DataFrames into a single DataFrame.
 - Reset the index for a clean structure.
#### Step 6 — Identify musical-value columns
 - Separate metadata columns (Composer, Title, Date, Measure, Beat) from the voice columns containing the actual n-gram data.
#### Step 7 — Reshape the data into long, tidy format
 - Melt the DataFrame so that each row represents: one voice in one location producing one n-gram.
 - Drop rows where no n-gram is present.
#### Step 8 — Clean and standardize the output
 - Rename the melted columns to `Voice` and `Ngram`.
 - Convert raw tuple-based n-grams into human-readable format with `_convertTuple`.
#### Final Output
 - `har_corpus_melted` — a tidy dataset where each row encodes: Composer | Title | Date | Measure | Beat | Voice → harmonic n-gram


In [13]:
func1 = ImportedPiece.harmonic
list_of_dfs = corpus.batch(func=func1, kwargs={'kind': 'd'}, metadata=True)
func2 = ImportedPiece.ngrams
list_of_harmonic_ngrams = corpus.batch(func=func2, kwargs={'n': 4, 'df': list_of_dfs})
func3 = ImportedPiece.detailIndex
list_of_detail_index = corpus.batch(func=func3, kwargs={'offset':False,'df': list_of_harmonic_ngrams}, metadata=True)
func4 = ImportedPiece.numberParts
list_of_np = corpus.batch(func=func4, kwargs={'df': list_of_detail_index}, metadata=True)
har_corpus = pd.concat(list_of_np)
har_corpus = har_corpus.reset_index()
val_vars = [col for col in har_corpus.columns if col not in ['Composer', 'Title', 'Date', 'Measure', 'Beat']]
har_corpus_melted = har_corpus.melt(id_vars = ['Composer', 'Title', 'Date', 'Measure', 'Beat'], value_vars = val_vars).dropna(subset = 'value')
har_corpus_melted = har_corpus_melted.rename(columns= {'variable': 'Voice', 'value': 'Ngram'})
har_corpus_melted['Ngram'] = har_corpus_melted['Ngram'].apply(_convertTuple)
har_corpus_melted

,Composer,Title,Date,Measure,Beat,Voice,Ngram
191,William Horwood,Gaude Flore Virginali,None,29.0,1.0,5_4,5_5_8_10
193,William Horwood,Gaude Flore Virginali,None,29.0,3.0,5_4,5_8_10_5
195,William Horwood,Gaude Flore Virginali,None,30.0,1.0,5_4,8_10_5_4
200,William Horwood,Gaude Flore Virginali,None,30.0,3.0,5_4,10_5_4_3
203,William Horwood,Gaude Flore Virginali,None,31.0,1.0,5_4,5_4_3_1
...,...,...,...,...,...,...,...
540478,Hugh Kellyk,Gaude Flore Virginali,None,274.0,1.0,6_1,17_16_15_14
540480,Hugh Kellyk,Gaude Flore Virginali,None,274.0,2.0,6_1,16_15_14_13
540481,Hugh Kellyk,Gaude Flore Virginali,None,274.0,2.5,6_1,15_14_13_12
540482,Hugh Kellyk,Gaude Flore Virginali,None,274.0,3.0,6_1,14_13_12_13


## 2.2 Matching Full N-Gram
This cell allows the user to search for a specific harmonic pattern in the corpus

 - The cell begins by defining `my_pattern` - the pattern of interest for the user (e.g., `'8_8_8_8'`).
 - The DataFrame is then filtered so that only rows who n-gram matches `my pattern` are displayed across the corpus.
 - These matches are then sorted to make them easier to read and interpret.
 - After sorting, `.head()` is used to display the first few results, giving a quick, organized snapshot of where this particular pattern occurs in the corpus. `.head()` can be commented out to display all results.

In [14]:
my_pattern = '8_8_8_8'

filtered_df = har_corpus_melted[har_corpus_melted['Ngram'] == my_pattern]

sorted_df_custom = filtered_df.sort_values(
    by=['Title', 'Measure', 'Beat', 'Voice'],
    ascending=[True, True, True, True]  # Sort Measure in descending order
)

sorted_df_custom.head()

,Composer,Title,Date,Measure,Beat,Voice,Ngram
247698,William Cornysh,Ave Maria Mater Dei,None,39.0,4.000000,2_1,8_8_8_8
133228,Robert Fayrfax,E57 Ave Lumen Gratie,None,49.0,2.333333,4_2,8_8_8_8
133229,Robert Fayrfax,E57 Ave Lumen Gratie,None,50.0,1.000000,4_2,8_8_8_8
133230,Robert Fayrfax,E57 Ave Lumen Gratie,None,50.0,1.333333,4_2,8_8_8_8
133240,Robert Fayrfax,E57 Ave Lumen Gratie,None,53.0,1.000000,4_2,8_8_8_8


### 2.2.1 Count occurrences of `my_pattern`

In [15]:
count = filtered_df.shape[0]
print(f"Total occurrences of '{my_pattern}':", count)

Total occurrences of '8_8_8_8': 48


## 2.3 Matching Any Substring

This code searches the harmonic corpus for any n-gram that contains a shorter interval pattern stored in a newly-defined `my_pattern`. Rather than requiring an exact match, `str.contains()` identifies all n-grams in which the specified shape appears as part of a longer pattern. Once these partial matches are collected, the results are sorted by piece title, then by measure, beat, and voice, making it easy to see where these occurrences appear in the music. In this context, `my_pattern` should represent a substring shorter than the full n-gram so that it can successfully match within longer harmonic patterns.

In [16]:
my_pattern = '7'

filtered_df = har_corpus_melted[har_corpus_melted['Ngram'].str.contains(my_pattern)]

# Sort by multiple columns
sorted_df_custom = filtered_df.sort_values(
    by=['Title', 'Measure', 'Beat', 'Voice'],
    ascending=[True, True, True, True]  # Sort Measure in descending order
)

sorted_df_custom

,Composer,Title,Date,Measure,Beat,Voice,Ngram
195943,William Cornysh,Ave Maria Mater Dei,None,10.0,2.00,3_2,5_6_8_7
221729,William Cornysh,Ave Maria Mater Dei,None,11.0,2.75,3_1,8_5_6_7
221730,William Cornysh,Ave Maria Mater Dei,None,11.0,3.00,3_1,5_6_7_8
221731,William Cornysh,Ave Maria Mater Dei,None,11.0,4.00,3_1,6_7_8_6
221732,William Cornysh,Ave Maria Mater Dei,None,11.0,4.50,3_1,7_8_6_6
...,...,...,...,...,...,...,...
67018,John Browne,Stabat mater dolorosa,None,239.0,1.50,5_2,8_8_7_6
479453,John Browne,Stabat mater dolorosa,None,239.0,2.50,6_3,14_15_19_17
479454,John Browne,Stabat mater dolorosa,None,239.0,2.75,6_3,15_19_17_17
67023,John Browne,Stabat mater dolorosa,None,239.0,3.00,5_2,8_7_6_5


## 3.1 Contrapuntal nGrams

The following section allows for the identification and searching of Contrapuntal modules across a corpus

The first cell builds contrapuntal modules of `n` length (a contrapuntal n-gram) for every piece in the corpus. A location and voice information information is then attached before this information is then organised for easier viewing.

#### Step 1 - Build contrapuntal n-grams for each piece.
 - Uses `ImportedPiece.ngrams` from CRIM Intervals to create n-interval sequences (modules).
 - `interval_settings=('d', True, False)` tells CRIM how to encode the intervals (e.g. diatonic, with particular settings for direction/quality). [ALTERNATIVE SEARCH OPTIONS ARE AVAIBLE]
 - `offsets='last'` anchors each module to the time position of its last note.
 - `corpus.batch` applies this to every piece, producing a list of DataFrames, one per piece, containing all their 8-note modules.

#### Step 2 - Add positional details
 - `detailIndex` adds measure numbers, beat positions, and other index information to each module.
 - The output is another list of DataFrames, now with both modules and their precise locations in the piece.

#### Step 3 - Number and label the voices
 - `numberParts` assigns voice/part numbers so each module can be tied to a specific line in the texture.
 - `metadata=True` adds composer, title, date, etc., so every row “knows” which piece it belongs to.

#### Step 4 - Combine all pieces into a single corpus DataFrame
 - Stacks all per-piece DataFrames into one corpus-wide table: `contra_corpus`.
 - reset_index() cleans up the index after concatenation.

#### Step 5 - Separate metadata from musical values
 - Identifies which columns hold musical content (the modules/voices) versus metadata (composer, title, date, measure, beat).

#### Step 6 - Reshape to a tidy, long-format corpus of modules
 - `melt` converts the wide table (many voice columns) into a long table where:
  - each row = one voice at one location with one n-note interval module.
 - Drops rows with missing values.
 - Renames columns so:
  -  `Voice` = the voice/part label
  -  `Ngram` = the n-interval contrapuntal module
 - The final `contra_corpus_melted` dataframe is a tidy corpus of contrapuntal modules, ready for searching and further analysis.

In [17]:
func = ImportedPiece.ngrams
list_of_modules = corpus.batch(func=func, kwargs={'n': 4, 'interval_settings': ('d', True, False), 'offsets': 'last'}, metadata=False)

func2 = ImportedPiece.detailIndex
list_of_details = corpus.batch(func=func2, kwargs={'offset': False, 'df': list_of_modules}, metadata=False)

# list_of_detail_index = corpus.batch(func=func3, kwargs={'offset':False,'df': list_of_details}, metadata=False)
func3 = ImportedPiece.numberParts
list_of_np = corpus.batch(func=func3, kwargs={'df': list_of_details}, metadata=True)

contra_corpus = pd.concat(list_of_np)
contra_corpus = contra_corpus.reset_index()

val_vars = [col for col in contra_corpus.columns if col not in ['Composer', 'Title', 'Date', 'Measure', 'Beat']]

contra_corpus_melted = contra_corpus.melt(id_vars = ['Composer', 'Title', 'Date', 'Measure', 'Beat'], value_vars = val_vars).dropna(subset = 'value')
contra_corpus_melted = contra_corpus_melted.rename(columns= {'variable': 'Voice', 'value': 'Ngram'})
contra_corpus_melted

,Composer,Title,Date,Measure,Beat,Voice,Ngram
168,William Horwood,Gaude Flore Virginali,None,30.0,3.00,5_4,"5_1, 5_-4, 8_1, 3"
171,William Horwood,Gaude Flore Virginali,None,31.0,1.00,5_4,"5_-4, 8_1, 3_5, 5"
172,William Horwood,Gaude Flore Virginali,None,31.0,1.50,5_4,"8_1, 3_5, 5_Held, 4"
173,William Horwood,Gaude Flore Virginali,None,31.0,2.00,5_4,"3_5, 5_Held, 4_Held, 3"
175,William Horwood,Gaude Flore Virginali,None,31.0,2.50,5_4,"5_Held, 4_Held, 3_Held, 1"
...,...,...,...,...,...,...,...
485589,Hugh Kellyk,Gaude Flore Virginali,None,274.0,3.00,6_1,"3_Held, 2_Held, 8_2, 7"
485590,Hugh Kellyk,Gaude Flore Virginali,None,274.0,3.50,6_1,"2_Held, 8_2, 7_Held, 6"
485591,Hugh Kellyk,Gaude Flore Virginali,None,274.0,3.75,6_1,"8_2, 7_Held, 6_Held, 5"
485592,Hugh Kellyk,Gaude Flore Virginali,None,274.0,4.00,6_1,"7_Held, 6_Held, 5_Held, 6"


## 3.2 Matching Full N-Gram
 This cell allows the user to search the contrapuntal corpus for a specific n-note interval module. `n` must be the same value defined in 3.1.

 - The cell begins by defining `my_pattern`, which stores the exact interval sequence the user wants to locate (e.g., `'3_2, 3_2, 8_2, 6'`).
 - The DataFrame `contra_corpus_melted` is then filtered so that only rows whose `Ngram` column exactly matches this pattern are shown. This produces a subset of the corpus displaying every occurrence of the module across all pieces and voices.
 - The matching rows are sorted by Title, Measure, Beat, and Voice to present the results in a clear and musically logical order.
 - Finally, the sorted results are displayed. Using `.head()` (which is currently commented out) would show only the first few matches; leaving it commented shows the full set of results.


In [18]:
my_pattern = '3_2, 3_2, 8_2, 6'
filtered_df = contra_corpus_melted[contra_corpus_melted['Ngram'] == my_pattern]

# Sort by multiple columns
sorted_df_custom = filtered_df.sort_values(
    by=['Title', 'Measure', 'Beat', 'Voice'],
    ascending=[True, True, True, True]  # Sort Measure in descending order
)

sorted_df_custom#.head()

,Composer,Title,Date,Measure,Beat,Voice,Ngram
12977,John Browne,Stabat mater dolorosa,None,116.0,2.0,5_4,"3_2, 3_2, 8_2, 6"
106108,John Browne,Stabat mater dolorosa,None,174.0,4.5,4_3,"3_2, 3_2, 8_2, 6"


## 3.3 Matching Any Substring (Contrapuntal Modules)

This code searches the contrapuntal corpus for any `n`-interval module that contains a shorter interval sequence stored in the newly defined `my_pattern`. Instead of requiring a full-module match, `str.contains()` identifies all interval modules in which the specified sequence appears as part of a longer pattern. Once these partial matches are retrieved, the results are sorted by title, then by measure, beat, and voice so that their occurrences can be read in a clear and musically logical order. In this context, `my_pattern` should represent a substring shorter than the full 8-interval module to ensure that it can be found naturally within longer contrapuntal sequences.

In [19]:
# search for substring pattern

my_pattern = '3_2, 3_2, 8_2, 6'
filtered_df = contra_corpus_melted[contra_corpus_melted['Ngram'].str.contains(my_pattern)]

# Sort by multiple columns
sorted_df_custom = filtered_df.sort_values(
    by=['Title', 'Measure', 'Beat', 'Voice'],
    ascending=[True, True, True, True]  # Sort Measure in descending order
)

sorted_df_custom

,Composer,Title,Date,Measure,Beat,Voice,Ngram
12977,John Browne,Stabat mater dolorosa,None,116.0,2.0,5_4,"3_2, 3_2, 8_2, 6"
106108,John Browne,Stabat mater dolorosa,None,174.0,4.5,4_3,"3_2, 3_2, 8_2, 6"


In [20]:
!jupyter nbconvert --to html "Eton Recon Aid for Appendix.ipynb"

[NbConvertApp] Converting notebook Eton Recon Aid for Appendix.ipynb to html
[NbConvertApp] Writing 389401 bytes to Eton Recon Aid for Appendix.html
